# Results Investigation

### Objective

The purpose of this investigation is to analyze the validation reports generated by the data validation pipeline, verify intentionally introduced inconsistencies, and identify unexpected patterns in the source data.

The investigation focuses on:
- Verifying that intentionally introduced data quality issues are detected correctly.
- Identifying downstream effects of broken references between related tables.
- Investigating recurring validation failures and distinguishing confirmed issues from patterns that require further review.

The findings are used to assess the validation rules and document the observed data quality issues. Source data are not modified during this investigation.

### Patients Reports

In [48]:
import pandas as pd
from prepare_data import load
from utils import check_datatype
dict_reports = load('reports')

In [49]:
patients = dict_reports['patients']
display(patients)

,DF,Index,Id,BIRTHDATE,FIRST,LAST,GENDER,Reason
0,patients,9,NaN,1972-09-11,Merlene950,Thompson596,F,Id is NaN
1,patients,108,7132442e-b0b9-a886-e28d-f83700e5a3bd,1988-12-25,Carman694,Bergnaum523,F,Id is duplicated


In [50]:
print('Number of rows in the report: ', len(patients))
print('Number of unique records in the report: ', len(patients['Index'].unique()))

Number of rows in the report:  2
Number of unique records in the report:  2


The found invalid records correspond to the purposeful inconsistencies at index 9 and 108.

| Inconsistency | DataFrame | Index | What is wrong |
|---|---|---|---|
| 1 | patients | 9 | Missing Id |
| 2 | patients | 108 (last appended row) | Duplicate Id |


----

### Encounters Report

In [51]:
encounters = dict_reports['encounters']
display(encounters)

,DF,Index,Id,PATIENT,START,STOP,ENCOUNTERCLASS,DESCRIPTION,REASONCODE,REASONDESCRIPTION,Reason
0,encounters,19,8b9df7af-24b3-66ee-d46c-520a00b76fa8,8b9df7af-24b3-66ee-afeb-2734ebcbd33a,2017-03-26T03:41:03Z,2017-03-26T03:56:03Z,wellness,Well child visit (procedure),NaN,NaN,PATIENT Id references non-existent Id
1,encounters,331,f5749532-3295-ad01-d458-6563c570663c,f5749532-3295-ad01-d9b5-932c997e7a01,1990-11-05T07:20:24Z,1990-11-05T08:09:10Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
2,encounters,332,f5749532-3295-ad01-e596-ecdf3cf1e767,f5749532-3295-ad01-d9b5-932c997e7a01,2000-11-20T07:20:24Z,2000-11-20T08:03:53Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
3,encounters,333,f5749532-3295-ad01-ff01-45763dd7c52b,f5749532-3295-ad01-d9b5-932c997e7a01,2000-11-27T07:20:24Z,2000-11-27T08:23:54Z,ambulatory,Encounter for problem (procedure),271737000.0,Anemia (disorder),PATIENT Id references non-existent Id
4,encounters,334,f5749532-3295-ad01-64d4-ecb86b7d8653,f5749532-3295-ad01-d9b5-932c997e7a01,2003-11-24T07:20:24Z,2003-11-24T08:03:47Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
5,encounters,335,f5749532-3295-ad01-f559-54b5bf1bdefd,f5749532-3295-ad01-d9b5-932c997e7a01,2012-09-17T07:20:24Z,2012-09-17T08:15:07Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
6,encounters,336,f5749532-3295-ad01-b198-0b8422f92d43,f5749532-3295-ad01-d9b5-932c997e7a01,2012-10-08T07:20:24Z,2012-10-08T08:25:38Z,ambulatory,Encounter for problem (procedure),37320007.0,Loss of teeth (disorder),PATIENT Id references non-existent Id
7,encounters,337,f5749532-3295-ad01-245e-b4c477ee898a,f5749532-3295-ad01-d9b5-932c997e7a01,2014-09-22T07:20:24Z,2014-09-22T07:55:30Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
8,encounters,338,f5749532-3295-ad01-c53e-0cfd46599b2c,f5749532-3295-ad01-d9b5-932c997e7a01,2015-09-09T08:29:31Z,2015-09-09T08:44:31Z,outpatient,Consultation for treatment (procedure),389095005.0,Contraception care (regime/therapy),PATIENT Id references non-existent Id
9,encounters,339,f5749532-3295-ad01-0f8c-1aea90e943c6,f5749532-3295-ad01-d9b5-932c997e7a01,2016-09-03T08:29:31Z,2016-09-03T08:44:31Z,outpatient,Consultation for treatment (procedure),389095005.0,Contraception care (regime/therapy),PATIENT Id references non-existent Id


In [52]:
print('Number of rows in the report: ', len(encounters))
print('Number of unique records in the report: ', len(encounters['Index'].unique()))

Number of rows in the report:  39
Number of unique records in the report:  39


In [53]:
invalid_patient_encounters = encounters[encounters['Reason'] == 'PATIENT Id references non-existent Id']
print(invalid_patient_encounters['PATIENT'].value_counts())

PATIENT
f5749532-3295-ad01-d9b5-932c997e7a01    36
8b9df7af-24b3-66ee-afeb-2734ebcbd33a     1
Name: count, dtype: int64


Inconsistency \#1 (deleting Patient Id 'f5749532-3295-ad01-d9b5-932c997e7a01') in patients table translated to the encounters table because  the encounters were referencing to a non-existent Id in patients table. This corresponds to 36 records. 


Inconsistency \#4 (invalidating Patient Id '8b9df7af-24b3-66ee-afeb-2734ebcbd33a') in encounters table was caught as 1 record.

In [54]:
other_encounters = encounters[encounters['Reason'] != 'PATIENT Id references non-existent Id']
display(other_encounters)

,DF,Index,Id,PATIENT,START,STOP,ENCOUNTERCLASS,DESCRIPTION,REASONCODE,REASONDESCRIPTION,Reason
37,encounters,1,cc3eac4a-34a7-9a15-1dc5-a353457420c6,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,2006-03-27T09:29:36Z,wellness,General examination of patient (procedure),NaN,NaN,START datetime is null
38,encounters,0,cc3eac4a-34a7-9a15-13f3-57f4a6b934f9,cc3eac4a-34a7-9a15-1522-8087135631b7,1996-03-11T09:21:33Z,1996-03-11T08:35:36Z,wellness,General examination of patient (procedure),NaN,NaN,STOP datetime is before START datetime


| Inconsistency | DataFrame | Index | What is wrong |
|---|---|---|---|
| 3 | encounters | 1 | Missing START |
| 4 | encounters | 19 | Invalid PATIENT ID |
| 5 | encounters | 0 | START and STOP switched |

Inconsistency \#3 was caught and resulted in failure of 2 validation criteria of NaN value – invalid datetime and null datetime value, at Index 1.

Inconsistency \#5 was also caught and corresponds to the switched STOP and START datetime of a record at Index 0.

---

### Observations Report

In [55]:
observations = dict_reports['observations']
display(observations.head(10))

,DF,Index,DATE,PATIENT,ENCOUNTER,CATEGORY,DESCRIPTION,VALUE,UNITS,TYPE,Reason
0,observations,2657,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,laboratory,Hemoglobin A1c/Hemoglobin.total in Blood,6.0,%,numeric,PATIENT Id references non-existent Id
1,observations,2658,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Body Height,164.3,cm,numeric,PATIENT Id references non-existent Id
2,observations,2659,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Pain severity - 0-10 verbal numeric rating [Sc...,1.0,{score},numeric,PATIENT Id references non-existent Id
3,observations,2660,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Body Weight,81.7,kg,numeric,PATIENT Id references non-existent Id
4,observations,2661,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Body mass index (BMI) [Ratio],30.3,kg/m2,numeric,PATIENT Id references non-existent Id
5,observations,2662,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Diastolic Blood Pressure,79.0,mm[Hg],numeric,PATIENT Id references non-existent Id
6,observations,2663,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Systolic Blood Pressure,117.0,mm[Hg],numeric,PATIENT Id references non-existent Id
7,observations,2664,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Heart rate,72.0,/min,numeric,PATIENT Id references non-existent Id
8,observations,2665,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,vital-signs,Respiratory rate,16.0,/min,numeric,PATIENT Id references non-existent Id
9,observations,2666,2016-09-26T07:20:24Z,f5749532-3295-ad01-d9b5-932c997e7a01,f5749532-3295-ad01-0049-abe9f9382d17,laboratory,Glucose [Mass/volume] in Blood,91.6,mg/dL,numeric,PATIENT Id references non-existent Id


In [56]:
print('Number of rows in the report: ', len(observations))
print('Number of unique records in the report: ', len(observations['Index'].unique()))

Number of rows in the report:  4850
Number of unique records in the report:  4445


In [57]:
print(observations['Reason'].value_counts())

Reason
CATEGORY is null                                     3060
TYPE does not correspond to the datatype of VALUE    1002
PATIENT Id references non-existent Id                 397
References an ENCOUNTER which was not validated       390
ENCOUNTER Id references non-existent Id                 1
Name: count, dtype: int64


In [11]:
deleted_Id = observations[observations['Reason'] == 'PATIENT Id references non-existent Id']
deleted_Id['PATIENT'].value_counts()

PATIENT
f5749532-3295-ad01-d9b5-932c997e7a01    397
Name: count, dtype: int64

Again, inconsistency \#1 (deleting Patient Id 'f5749532-3295-ad01-d9b5-932c997e7a01') in patients table translated even to the observations table because the observations were referencing to a non-existent Id in patients table. This corresponds to 397 caught records. 

In [12]:
deleted_encounter_Id = observations[observations['Reason'] == 'ENCOUNTER Id references non-existent Id']
display(deleted_encounter_Id)

,DF,Index,DATE,PATIENT,ENCOUNTER,CATEGORY,DESCRIPTION,VALUE,UNITS,TYPE,Reason
397,observations,965,2020-10-20T07:32:50Z,65cbad6a-eb9d-420e-9bab-b4f9c6aab7db,65cbad6a-eb9d-420e-0c28-4eb837f969bz,survey,Generalized anxiety disorder 7 item (GAD-7) to...,3.0,{score},numeric,ENCOUNTER Id references non-existent Id


The purposeful inconsistency \#6 (invalidating Encounter Id '65cbad6a-eb9d-420e-0c28-4eb837f969bc') was found.

| Inconsistency | DataFrame | Index | What is wrong |
|---|---|---|---|
| 6 | observations | 965 | Invalid ENCOUNTER ID |

In [30]:
invalid_references_Id = observations[observations['Reason'] == 'References an ENCOUNTER which was not validated']
unique_ones = invalid_references_Id['ENCOUNTER'].unique()
display(unique_ones)

<StringArray>
['8b9df7af-24b3-66ee-d46c-520a00b76fa8',
 'f5749532-3295-ad01-0049-abe9f9382d17',
 'f5749532-3295-ad01-7458-0a7edc951fd9',
 'f5749532-3295-ad01-6506-a9bcd107ca02',
 'f5749532-3295-ad01-7a44-711db61d3a16',
 'f5749532-3295-ad01-ad12-bd0e89f74c09',
 'f5749532-3295-ad01-1f5f-89549394d085',
 'f5749532-3295-ad01-6cbd-ec9547d6551c',
 'f5749532-3295-ad01-b7a2-ddd897c9e7da',
 'f5749532-3295-ad01-8285-7d02efa5284b']
Length: 10, dtype: str

In [61]:
encounters[encounters['Id'].isin(list(unique_ones))]

,DF,Index,Id,PATIENT,START,STOP,ENCOUNTERCLASS,DESCRIPTION,REASONCODE,REASONDESCRIPTION,Reason
0,encounters,19,8b9df7af-24b3-66ee-d46c-520a00b76fa8,8b9df7af-24b3-66ee-afeb-2734ebcbd33a,2017-03-26T03:41:03Z,2017-03-26T03:56:03Z,wellness,Well child visit (procedure),NaN,NaN,PATIENT Id references non-existent Id
10,encounters,340,f5749532-3295-ad01-0049-abe9f9382d17,f5749532-3295-ad01-d9b5-932c997e7a01,2016-09-26T07:20:24Z,2016-09-26T08:03:42Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
13,encounters,343,f5749532-3295-ad01-7458-0a7edc951fd9,f5749532-3295-ad01-d9b5-932c997e7a01,2017-11-20T07:20:24Z,2017-11-20T07:56:56Z,outpatient,Encounter for check up (procedure),NaN,NaN,PATIENT Id references non-existent Id
16,encounters,346,f5749532-3295-ad01-6506-a9bcd107ca02,f5749532-3295-ad01-d9b5-932c997e7a01,2018-10-01T07:20:24Z,2018-10-01T07:53:25Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
20,encounters,350,f5749532-3295-ad01-7a44-711db61d3a16,f5749532-3295-ad01-d9b5-932c997e7a01,2020-10-05T07:20:24Z,2020-10-05T08:04:16Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
24,encounters,354,f5749532-3295-ad01-ad12-bd0e89f74c09,f5749532-3295-ad01-d9b5-932c997e7a01,2021-09-25T14:20:24Z,2021-09-25T14:35:24Z,ambulatory,Encounter for symptom (procedure),43878008.0,Streptococcal sore throat (disorder),PATIENT Id references non-existent Id
29,encounters,359,f5749532-3295-ad01-1f5f-89549394d085,f5749532-3295-ad01-d9b5-932c997e7a01,2022-09-12T07:20:24Z,2022-09-12T08:12:57Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
32,encounters,362,f5749532-3295-ad01-6cbd-ec9547d6551c,f5749532-3295-ad01-d9b5-932c997e7a01,2023-09-18T07:20:24Z,2023-09-18T08:07:46Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
33,encounters,363,f5749532-3295-ad01-b7a2-ddd897c9e7da,f5749532-3295-ad01-d9b5-932c997e7a01,2024-09-23T07:20:24Z,2024-09-23T08:02:05Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id
35,encounters,365,f5749532-3295-ad01-8285-7d02efa5284b,f5749532-3295-ad01-d9b5-932c997e7a01,2025-09-29T07:20:24Z,2025-09-29T08:12:00Z,wellness,General examination of patient (procedure),NaN,NaN,PATIENT Id references non-existent Id


In [65]:
invalid_references_Id['ENCOUNTER'].value_counts()

ENCOUNTER
f5749532-3295-ad01-7458-0a7edc951fd9    57
f5749532-3295-ad01-6cbd-ec9547d6551c    52
f5749532-3295-ad01-1f5f-89549394d085    45
f5749532-3295-ad01-8285-7d02efa5284b    44
f5749532-3295-ad01-0049-abe9f9382d17    43
f5749532-3295-ad01-7a44-711db61d3a16    43
f5749532-3295-ad01-6506-a9bcd107ca02    42
f5749532-3295-ad01-b7a2-ddd897c9e7da    40
8b9df7af-24b3-66ee-d46c-520a00b76fa8    23
f5749532-3295-ad01-ad12-bd0e89f74c09     1
Name: count, dtype: int64

In [66]:
invalid_references_Id['PATIENT'].value_counts()

PATIENT
f5749532-3295-ad01-d9b5-932c997e7a01    367
8b9df7af-24b3-66ee-afeb-2734ebcbd33c     23
Name: count, dtype: int64

367 records of patient with Id 'f5749532-3295-ad01-d9b5-932c997e7a01' were marked as 'References an ENCOUNTER which was not validated'.
They are down-stream failures because of the inconsistency \#1, where patient Id was deleted from patients table. Afterwards, the encounters in encounters table corresponding to this patient were not validated, making them invalidated even in observations table (367 rows of observations). 

23 records of patient with Id '8b9df7af-24b3-66ee-afeb-2734ebcbd33c' were marked as 'References an ENCOUNTER which was not validated'. hey are down-stream failures because of the inconsistency \#4, where encounter Id was changed in encounters table. As a result, the original encounter IDs stored in the observations table no longer matched the corresponding IDs in the encounters table.

**Overlap between validation issues**

The invalid observations show an overlap between the validation reasons PATIENT Id references non-existent Id and References an ENCOUNTER which was not validated.

Out of 390 observations marked with References an ENCOUNTER which was not validated, 367 also had the issue PATIENT Id references non-existent Id. The remaining 23 observations were affected by the changed encounter ID (inconsistency #4).

This confirms that the two validation reasons can identify the same observation record, even though they describe different issues.


In [67]:
invalid_patient_rows = set(
    observations[observations['Reason'] == 'PATIENT Id references non-existent Id']['Index']
)

invalid_encounter_rows = set(
    observations[observations['Reason'] == 'References an ENCOUNTER which was not validated']['Index']
)

print('PATIENT issue:', len(invalid_patient_rows))
print('ENCOUNTER issue:', len(invalid_encounter_rows))
print('Both issues:', len(invalid_patient_rows & invalid_encounter_rows))
print('Only PATIENT issue:', len(invalid_patient_rows - invalid_encounter_rows))
print('Only ENCOUNTER issue:', len(invalid_encounter_rows - invalid_patient_rows))
print('Total unique affected rows:', len(invalid_patient_rows | invalid_encounter_rows))

PATIENT issue: 397
ENCOUNTER issue: 390
Both issues: 367
Only PATIENT issue: 30
Only ENCOUNTER issue: 23
Total unique affected rows: 420


In [13]:
null_category = observations[(observations['Reason'] == 'CATEGORY is null') | (observations['Reason'] == 'CATEGORY is not allowed value')]
display(null_category)

,DF,Index,DATE,PATIENT,ENCOUNTER,CATEGORY,DESCRIPTION,VALUE,UNITS,TYPE,Reason
398,observations,238,2016-01-16T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,NaN,QALY,37.0,a,numeric,CATEGORY is null
399,observations,239,2017-01-16T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,NaN,QALY,38.0,a,numeric,CATEGORY is null
400,observations,240,2018-01-16T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,NaN,QALY,38.9,a,numeric,CATEGORY is null
401,observations,241,2019-01-16T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,NaN,QALY,39.9,a,numeric,CATEGORY is null
402,observations,242,2020-01-16T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,NaN,NaN,QALY,40.9,a,numeric,CATEGORY is null
...,...,...,...,...,...,...,...,...,...,...,...
3453,observations,68643,2021-10-11T11:39:55Z,bca1691f-8839-1d66-ed01-471134d55738,NaN,NaN,QOLS,0.5,{score},numeric,CATEGORY is null
3454,observations,68644,2022-10-11T11:39:55Z,bca1691f-8839-1d66-ed01-471134d55738,NaN,NaN,QOLS,0.5,{score},numeric,CATEGORY is null
3455,observations,68645,2023-10-11T11:39:55Z,bca1691f-8839-1d66-ed01-471134d55738,NaN,NaN,QOLS,0.5,{score},numeric,CATEGORY is null
3456,observations,68646,2024-10-11T11:39:55Z,bca1691f-8839-1d66-ed01-471134d55738,NaN,NaN,QOLS,0.5,{score},numeric,CATEGORY is null


In [14]:
display(null_category['DESCRIPTION'].value_counts())

DESCRIPTION
QALY    1020
DALY    1020
QOLS    1020
Name: count, dtype: int64

All of the caught invalid data with missing category are a specific set of the dataset concerning the quality and length of life, such as QALY, DALY and QOLS.

In [15]:
print(len(null_category['Index'].unique()))

3060


The failure concerns 3060 unique records which have CATEGORY field empty.

In [16]:
not_corresponding = observations[observations['Reason'] == 'TYPE does not correspond to the datatype of VALUE']
not_corresponding['DESCRIPTION'].value_counts()

DESCRIPTION
What was your best estimate of the total income of all family members from all sources  before taxes  in last year [PhenX]    904
LogMAR visual acuity left eye (observable entity)                                                                              44
LogMAR visual acuity right eye (observable entity)                                                                             44
Priority Level                                                                                                                  9
Diastolic Blood Pressure                                                                                                        1
Name: count, dtype: int64

In [17]:
not_corresponding['VALUE'].info()

<class 'pandas.Series'>
RangeIndex: 1002 entries, 3458 to 4459
Series name: VALUE
Non-Null Count  Dtype
--------------  -----
1002 non-null   str  
dtypes: str(1)
memory usage: 8.0 KB


In [18]:
not_corresponding['VALUE']

3458     42283
3459     42283
3460     42283
3461     42283
3462    eighty
         ...  
4455    105055
4456    105055
4457    105055
4458    105055
4459    105055
Name: VALUE, Length: 1002, dtype: str

In [19]:
not_corresponding['TYPE'].value_counts()

TYPE
text       1001
numeric       1
Name: count, dtype: int64

In [20]:
not_corresponding['VALUE'].apply(check_datatype).value_counts()

VALUE
numeric    1001
text          1
Name: count, dtype: int64

In [22]:
display(not_corresponding[not_corresponding['VALUE'] == 'eighty'])

,DF,Index,DATE,PATIENT,ENCOUNTER,CATEGORY,DESCRIPTION,VALUE,UNITS,TYPE,Reason
3462,observations,203,2026-02-09T08:35:36Z,cc3eac4a-34a7-9a15-1522-8087135631b7,cc3eac4a-34a7-9a15-0045-5d8e18d8dd42,vital-signs,Diastolic Blood Pressure,eighty,mm[Hg],numeric,TYPE does not correspond to the datatype of VALUE


One expected failure of corresponding VALUE-TYPE pair is a purposeful inconsistency \#7 at index 203 with string VALUE 'eighty' and TYPE 'numeric'. 


| Inconsistency | DataFrame | Index | What is wrong |
|---|---|---|---|
| 7 | observations | 203 | VALUE does not match expected numeric type |


Other 1001 records of observations with not corresponding VALUE-TYPE pair were unexpected. All of them describe numeric values which should have a TYPE as 'numeric'.

In [23]:
type_value_summary = (
    not_corresponding
    .groupby(
        ['DESCRIPTION', 'TYPE', 'UNITS'],
        dropna=False
    )
    .agg(
        count=('Index', 'size'),
        example_values=('VALUE', lambda x: list(x.head(3)))
    )
    .reset_index()
    .sort_values('count', ascending=False)
)

display(type_value_summary)

,DESCRIPTION,TYPE,UNITS,count,example_values
4,What was your best estimate of the total incom...,text,/a,904,"[42283, 42283, 42283]"
1,LogMAR visual acuity left eye (observable entity),text,{logmar},44,"[0, 0, 0]"
2,LogMAR visual acuity right eye (observable ent...,text,{logmar},44,"[0, 0, 0]"
3,Priority Level,text,{score},9,"[2, 1, 4]"
0,Diastolic Blood Pressure,numeric,mm[Hg],1,[eighty]


## Conclusion
Most of the reported records were caused by intentional inconsistencies that are explained in detail in README. Unexpected reported records included 3060 observation records with empty `'CATEGORY'` field and 1001 observation records with not corresponding VALUE-TYPE.